<a href="https://colab.research.google.com/github/trngdothuy/Market-Intelligence-Agent/blob/main/notebooks/01_Market_Intelligence_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚀 Talent Arena 2026
# Building a Market Intelligence Agent

## Welcome to the AI Agents Workshop!

<img src="https://img.shields.io/badge/MWC-2026-blue" /> <img src="https://img.shields.io/badge/Track-Market_Intelligence-green" /> <img src="https://img.shields.io/badge/Duration-90min-orange" />

---

### What You'll Build Today

A **Market Intelligence Agent** that given a stock ticker symbol (e.g. Apple -> AAPL):
- 📈 Fetches real-time stock prices
- 📰 Searches for relevant news when stocks move significantly
- 💡 Provides professional market analysis and insights
- 🤖 Uses the ReAct pattern for transparent reasoning
- ✅ Performs each step of the workflow correctly and in order

---

### What You'll Learn

- 🧠 **How AI Agents Work** — Beyond simple chatbots
- 🔄 **The ReAct Pattern** — Reasoning + Acting in loops
- 🛠️ **Tool Integration** — Giving agents real-world capabilities
- ✏️ **Prompt Engineering** — The art of steering agent behavior


---

## 📦 Part 1: Quick Setup

We'll install everything you need in one go. This takes about 60 seconds.

In [9]:
# 🔧 Automated Setup - Run this cell first!

import sys

# Download requirements and framework from GitHub
print("1️⃣ Downloading workshop files...")
!wget -O requirements.txt -q https://raw.githubusercontent.com/Sage/talent-arena-2026/master/requirements.txt
!wget -O lab.py -q https://raw.githubusercontent.com/Sage/talent-arena-2026/master/lab.py

print("\n2️⃣ Installing Python packages (this may take 30-60 seconds)...")
!pip install --upgrade pip --quiet
!pip install -r requirements.txt --quiet

print("✅ Setup complete! You're ready to build agents!")

1️⃣ Downloading workshop files...

2️⃣ Installing Python packages (this may take 30-60 seconds)...
✅ Setup complete! You're ready to build agents!


---

## ⚙️ Configure Your LLM Connection

Before we begin, you need to store two secrets so the notebook can talk to the LLM backend.

**📝 In Google Colab → click the 🔑 icon on the left sidebar → add:**

| Secret name | Value |
|-------------|-------|
| `LLM_SERVICE_URL` | The URL provided by the workshop facilitator |
| `PASSWORD` | The password provided by the workshop facilitator |


In [10]:
from google.colab import userdata
import os

key = userdata.get('LLM_SERVICE_URL')
password = userdata.get('PASSWORD')

if key and password:
    os.environ['LLM_SERVICE_URL'] = key
    os.environ['PASSWORD'] = password
    print("✅ API key and password configured from Colab userdata!")
else:
    print("⚠️ API key or password not found in Colab userdata. Please set them manually as environment variables.")

✅ API key and password configured from Colab userdata!


---

## 🧪 Quick Verification

Let's verify everything is working by testing the framework and tools.

In [11]:
# 🧪 Quick Verification

try:
    from lab import BaseAgent, Tools, test_all_tools
    print("\n🔬 Testing tools...\n")
    test_all_tools(verbose=True)
except Exception as e:
    print(f"❌ Something went wrong: {e}")
    print("💡 Re-run the setup cells above and try again.")



🔬 Testing tools...

🧪 Testing All Tools

✅ get_stock_price('AAPL')
   Price: $265.68 (+1.23%)

✅ search_news('technology')
   Found 3 articles

✅ scrape_hacker_news()
   Found 3 trending stories

✅ get_project_summary('Python')

✅ get_crypto_prices('BTC,ETH')
   BTC: $69705.75

✅ generate_chart(dict)
   URL: https://quickchart.io/chart?c=%7B%22type%22%3A%20%22bar%22%2...

✅ Tests passed: 6/6


---

# 🏗️ Build Your Market Intelligence Agent

Now it's your turn! You'll write a **system prompt** that turns a generic LLM into a stock-market analyst.

## 🎯 Your Mission

Build an agent that:
1. Fetches real-time stock prices
2. Checks if the move is significant (> 2%)
3. Searches for news **only** when the move is significant
4. Delivers a professional market analysis

---

## 🛠️ Available Tools

Tell your agent which tools to use and how. Each tool is invoked by name with a single string input:

| Tool | What It Does | Action Input Example |
|------|--------------|----------------------|
| `get_stock_price` | Current price, change, % change | `AAPL` |
| `search_news` | Recent news articles | `AAPL stock earnings news` |
| `scrape_hacker_news` | Top 3 trending Hacker News stories | *(no input needed)* |
| `get_project_summary` | Web search for any topic | `Apple Inc company overview` |
| `get_crypto_prices` | Cryptocurrency prices (Binance) | `BTC,ETH,SOL` |
| `generate_chart` | Create charts from numeric data (bar, line, pie, etc.) | `'{"data": {"BTC": 65000, "ETH": 3500}, "chart_type": "pie"}'` |

You don't need all of them — pick the ones that make sense for stock analysis.


---

## 💭 Prompt Engineering for Agents

Before coding, design your system prompt. Great prompts are the difference between chaos and intelligence.

**✅ DO:**
- Be explicit and specific ("ALWAYS", "NEVER", "MUST")
- Use conditional logic ("IF condition, THEN action")
- Provide examples of desired output
- Define step-by-step workflows
- Specify exact format requirements

**❌ DON'T:**
- Be vague ("analyze this", "look into that")
- Let the agent guess what to do
- Skip output format specification
- Forget to define stopping conditions

---

**💡 Tip**: The LLM follows instructions literally. If you don't specify it, the agent won't do it!


In [31]:
# 🏗️ Build Your Market Intelligence Agent
# Fill in the blanks in the system prompt below, then run this cell.

from lab import BaseAgent, Tools

tools = Tools()

class MyMarketAgent(BaseAgent):

    @property
    def system_prompt(self):
        return """You are a Professional Market Intelligence Analyst specialized in stock market analysis.

YOUR WORKFLOW (follow these steps IN ORDER):

Step 1: Fetch real time stock data
   - If there is no valid stock data, you must inform that symbol not found and tell the reason why as "this is likely because"
   - If there is no valid stock symbol, you must stop analyzing the price movements but tell them what to do next (like double check or look up the correct one)
   - Finally, inform them that it is unable to provide analysis without valid market data.
   - Do not print the thought process like "ERROR", "action" or "action input"

Step 2: Analyze price movements
   - If the change is less than 2%, you must state that the change is not significant
   -

Step 3: If appropriate, search for relevant news
   - If the change is not significant, you must skip searching for news
   -

Step 4: Summarize your findings
   -
   -

CRITICAL RULES:
- Follow the Thought → Action → Observation pattern for EACH step
- After receiving ALL necessary observations, your NEXT response MUST start with "Final Answer:"
-
-

CRITICAL FORMAT RULES:
- NEVER output just a Thought without Action/Action Input
-
-
-

OUTPUT FORMAT:

## 📈 [Company Name] ([SYMBOL]) - Market Analysis

**Current Price:** $[price]
**Change:** [change] ([percent_change]%)
**Status:** [Up/Down/Flat]

### 📊 Analysis:
[2-3 sentences about the price movement and what it means]

### 📰 Market Context:
[IF news was searched: Key news summary]
[IF no news: "No significant news events today - price movement is within normal range"]

### 💡 Key Takeaway:
[One sentence actionable insight]

---
*Data as of market close. Not financial advice.*

Remember: After getting all observations, start with "Final Answer:" - NEVER just output text!"""


# Create your agent
my_agent = MyMarketAgent(tools)
print(f"✅ Agent created succesfully")


✅ Agent created succesfully


---

## 🧪 Test Your Agent

Run the cell below to see your agent in action. If the output isn't what you expect, go back and refine your prompt!


In [25]:
# 🧪 Test Your Agent — change the symbol and re-run as many times as you like

test_symbol = "GOOGL"  # Try: TSLA, MSFT, PLTR, GOOGL, NVDA ...

print(f"📡 Analyzing {test_symbol}...\n")
for output in my_agent.run(test_symbol):
    print(output)

📡 Analyzing GOOGL...



KeyboardInterrupt: 

---

## 🔄 Iterate and Refine

Your first version won't be perfect - that's normal! Great agents are built through iteration.

### 🐛 Universal Debugging Patterns

**Issue: Agent doesn't use expected tools**
- ✅ Make tool usage explicit: "FIRST, use [tool_name]..."
- ✅ Make it mandatory: "You MUST use [tool] to..."
- ✅ Explain WHY to use each tool

**Issue: Agent repeats the same action**
- ✅ Add progression logic: "AFTER [action], move to [next_action]"
- ✅ Prevent loops: "DO NOT call [tool] more than once"
- ✅ Define clear stopping conditions

**Issue: Inconsistent or poor output format**
- ✅ Provide an exact template: "You MUST use this format:"
- ✅ Show a concrete example in the prompt
- ✅ Use strong language: "ALWAYS include...", "NEVER omit..."

**Issue: Agent does too much or too little**
- ✅ Add conditional logic: "IF [condition], THEN [action], ELSE [other_action]"
- ✅ Set clear boundaries: "ONLY search news if..."
- ✅ Define efficiency rules: "Limit to X tool calls"

**Issue: Agent "forgets" previous information**
- ✅ Add state tracking: "Keep track of all data you collect"
- ✅ Make memory explicit: "After each step, note: [key: value]"
- ✅ Reference previous observations: "Use the price you found earlier..."

**Issue: Wrong workflow or sequencing**
- ✅ Number your steps: "Step 1: [action]. Step 2: [action]..."
- ✅ Be MORE explicit about the entire workflow
- ✅ Define dependencies: "You can't do Y until you've done X"


---

## 🎯 Agent Testing & Optimization

Time to put your agent through its paces. Three scenarios, each targeting a specific capability. Run them in order — use the reference answers and checklists to judge your own output and decide where to improve.

| # | Scenario | What You're Testing |
|---|----------|---------------------|
| **1** | Small Price Movement | Basic workflow, tool efficiency, and output formatting |
| **2** | Invalid Symbol | Edge case handling and user-friendly error responses |
| **3** | Significant Price Movement | Conditional logic, multi-tool usage, and synthesis |

Each scenario includes:
- 🤖 **Your agent's live output** — what it actually does
- ✨ **A reference answer** — what a well-designed agent produces
- 📋 **A pass/fail checklist** — so you know exactly what to fix

> 💡 **A perfect first run is rare — and that's fine.** Spot a gap, update your prompt, re-run. That's the real skill you're building here.


---

### ✅ Scenario 1: Small Price Movement

A normal trading day — the stock has barely moved. This is the most common case and your agent's baseline test.

**Goal:** Confirm your agent executes the basic workflow cleanly and produces a professional, well-structured output — without doing unnecessary work.

**What we're checking:**
- Does it call `get_stock_price` exactly **once**?
- Does it correctly identify the movement as *not* significant (< 2%)?
- Does it **skip the news search**? *(Fetching news when nothing is happening is wasteful — good agents know when to stop)*
- Is the output clean, complete, and professional?

**The agent should:**
1. Fetch the stock price
2. See that the change is less than 2%
3. Skip the news search — it's not needed
4. Deliver a concise, well-formatted market summary


In [22]:
# 🧪 Scenario 1: Small Price Movement

import time

print(f"🎯 Testing: Scenario 1: Small Price Movement")
print(f"📝 Suggested Query: 'AAPL'\n")
print("=" * 70)
print("🤖 YOUR AGENT'S OUTPUT:")
print("=" * 70)

start = time.time()
try:
    for output in my_agent.run("AAPL"):
        print(output, end='')
    duration = time.time() - start

    print(f"\n{'=' * 70}")
    print(f"⏱️  Completed in {duration:.1f}s")
    print("=" * 70)
    print("\n👇 Compare your output to the reference answer below")

except ValueError as e:
    from lab import display_react_format_error
    display_react_format_error(e)


🎯 Testing: Scenario 1: Small Price Movement
📝 Suggested Query: 'AAPL'

🤖 YOUR AGENT'S OUTPUT:
Thought: I need to fetch real-time stock data for AAPL (Apple Inc.) as the first step in my workflow.
Action: get_stock_price
Action Input: AAPL
Observation: {'symbol': 'AAPL', 'price': 265.94, 'change': 3.5, 'percent_change': 1.33, 'market_status': 'open'}
Final Answer: ## 📈 Apple Inc. (AAPL) - Market Analysis

**Current Price:** $265.94
**Change:** +3.50 (+1.33%)
**Status:** 📈 Up

### 📊 Analysis:
AAPL is trading up **1.33%** today, which is **not a significant movement** as it falls below the 2% threshold for notable price action. The modest gain of $3.50 suggests routine market activity without any major catalysts driving the stock. This type of movement is considered normal daily fluctuation within Apple's typical trading range.

### 📰 Market Context:
No significant news events today — price movement is within normal range. The sub-2% change does not warrant further investigation into spec

#### ✨ Reference Answer

> **Note:** Prices shown are examples — your live data will differ. Focus on **format**, **structure**, and **quality**, not the exact numbers.

```
## 📈 Tesla (TSLA) - Market Analysis

**Current Price:** $248.10
**Change:** +$1.93 (+0.78%)
**Status:** Up

### 📊 Analysis:
Tesla is trading slightly higher with a 0.78% gain today. This is
within normal daily variance and does not signal a significant market
event. No further research is warranted.

### 📰 Market Context:
No significant news events today - price movement is within normal range.

### 💡 Key Takeaway:
Quiet session for Tesla — no action needed at current movement levels.

---
*Data as of market close. Not financial advice.*
```

---

#### 📋 Pass / Fail Checklist

**Your agent passes this scenario if:**

- ✅ Called `get_stock_price` exactly **once**
- ✅ Correctly identified the move as **less than 2%**
- ✅ **Did NOT call** `search_news` — skipping it here is the right decision
- ✅ Output includes: current price, change amount, and change percentage
- ✅ Includes a brief analysis — not just raw numbers
- ✅ Tone is professional and the format is clean
- ✅ Finished in **2–3 ReAct iterations**

> ⚠️ **If your agent searched for news anyway**, add a clear condition to your prompt.


---

### ⚠️ Scenario 2: Invalid Stock Symbol

A user made a typo — or just tested your agent with a nonsense ticker. Edge cases like this are important to consider. How does your agent respond?

**Goal:** Verify your agent handles bad input gracefully — no crashes, no confusion, just a calm and helpful response that guides the user forward.

**What we're checking:**
- Does it attempt `get_stock_price` with the invalid symbol?
- Does it recognize that no valid data was returned?
- Does it **stop** — without retrying or calling unnecessary tools?
- Does it explain the issue clearly and offer a practical next step?

**The agent should:**
1. Attempt to fetch data for `NKJL`
2. Receive an error or empty result
3. Recognize the symbol is invalid — no further tool calls needed
4. Respond with a clear, helpful message — no technical jargon, no panic


In [32]:
# 🧪 Scenario 2: Invalid Stock Symbol

print(f"🎯 Testing: Scenario 2: Invalid Stock Symbol")
print(f"📝 Suggested Query: 'NKJL'\n")
print("=" * 70)
print("🤖 YOUR AGENT'S OUTPUT:")
print("=" * 70)

start = time.time()
try:
    for output in my_agent.run("NKJL"):
        print(output, end='')
    duration = time.time() - start

    print(f"\n{'=' * 70}")
    print(f"⏱️  Completed in {duration:.1f}s")
    print("=" * 70)
    print("\n👇 Compare your output to the reference answer below")

except ValueError as e:
    from lab import display_react_format_error
    display_react_format_error(e)


🎯 Testing: Scenario 2: Invalid Stock Symbol
📝 Suggested Query: 'NKJL'

🤖 YOUR AGENT'S OUTPUT:
Thought: The user wants stock analysis for the symbol "NKJL". Let me fetch the real-time stock data for this symbol first.


ERROR:yfinance:$NKJL: possibly delisted; no price data found  (period=5d) (Yahoo error = "No data found, symbol may be delisted")


Action: get_stock_price
Action Input: NKJL
Observation: No data found for 'NKJL'. The market may be closed or the symbol is invalid.
Final Answer: ## ⚠️ Symbol Not Found: NKJL

**Status:** Invalid or Unrecognized Stock Symbol

---

### 🔍 What Happened:
No valid stock data could be retrieved for the symbol **"NKJL"**. This is likely because:

- **The symbol does not exist** on major exchanges (NYSE, NASDAQ, etc.)
- It may be a **misspelling** of an existing ticker symbol
- It could be a **delisted or inactive** security

---

### 📋 What To Do Next:

1. **Double-check the symbol** — Verify the correct ticker on financial sites like:
   - [Yahoo Finance](https://finance.yahoo.com)
   - [Google Finance](https://google.com/finance)
   - [MarketWatch](https://www.marketwatch.com)

2. **Did you mean one of these?**
   | You Typed | Possible Match |
   |-----------|---------------|
   | NKJL | **NKE** (Nike, Inc.) |
   | NKJL | **NKL** (Check exchange) |
   | NKJL | **NKLA** (Nikola Corporatio

#### ✨ Reference Answer

> **Note:** Focus on **error handling**, **clarity**, and **helpfulness** — not data accuracy.

```
⚠️ Symbol Not Found: NKJL

No market data is available for the ticker "NKJL".

This is likely because:
• The symbol is misspelled or does not exist
• The company may not be publicly traded
• The ticker may have been retired due to a merger or delisting

What to do next:
→ Double-check the symbol (e.g., Apple → AAPL, Tesla → TSLA)
→ Look up the correct ticker at finance.yahoo.com or Google Finance

*Unable to provide analysis without valid market data.*
```

---

#### 📋 Pass / Fail Checklist

**Your agent passes this scenario if:**

- ✅ Attempted `get_stock_price` with the provided symbol
- ✅ Recognized that the response contained no valid data
- ✅ **Did NOT retry** the same call or invoke unnecessary tools
- ✅ States clearly that the symbol was not found — no raw errors or technical output
- ✅ Offers suggestions to help the user
- ✅ Tone is calm and professional — not apologetic or alarming

> ⚠️ **You may have to add clear instructions or a standard answer for this edge case**


---

### 🔥 Scenario 3: Significant Price Movement

A stock has moved more than 2% — something is happening in the market and raw price data isn't enough. Time to investigate.

**Goal:** Verify your agent correctly detects significant movement, automatically triggers a news search, and delivers a connected analysis that explains *why* the price moved — not just *that* it moved. A great agent looks beyond the stock itself: macro conditions, geopolitics, and market-wide events often matter more than company-specific news.

**What we're checking:**
- Does it fetch the price and identify a move **greater than 2%**?
- Does it **automatically trigger** `search_news`? *(This is the conditional logic from your prompt — the key test)*
- Does it connect the news to the price movement, rather than listing them separately?
- Does it consider **macro and geopolitical factors** — not just company news?
- Is the final output a cohesive analysis — not a data dump?

**The agent should:**
1. Fetch the stock price
2. Detect that the change exceeds 2% — this triggers the next step
3. Search for relevant news to understand *why*
4. Synthesize price data, news, and broader context into a connected, insightful analysis



In [ ]:
# 🧪 Scenario 3: Significant Price Movement

print(f"🎯 Testing: Scenario 3: Significant Price Movement")
print(f"📝 Suggested Query: 'AAPL'\n")
print("=" * 70)
print("🤖 YOUR AGENT'S OUTPUT:")
print("=" * 70)

start = time.time()
try:
    for output in my_agent.run("AAPL"):
        print(output, end='')
    duration = time.time() - start

    print(f"\n{'=' * 70}")
    print(f"⏱️  Completed in {duration:.1f}s")
    print("=" * 70)
    print("\n👇 Compare your output to the reference answer below")

except ValueError as e:
    from lab import display_react_format_error
    display_react_format_error(e)


🎯 Testing: Scenario 3: Significant Price Movement
📝 Suggested Query: 'AAPL'

🤖 YOUR AGENT'S OUTPUT:
Thought: I need to fetch real-time stock data for Apple Inc. (AAPL) first.
Action: get_stock_price
Action Input: AAPL
Observation: {'symbol': 'AAPL', 'price': 264.18, 'change': -8.77, 'percent_change': -3.21, 'market_status': 'closed'}


#### ✨ Reference Answer

> **Note:** Prices and news shown are examples. Focus on **conditional logic**, **synthesis**, and **depth of analysis** — not the exact data.

```
## 📈 Apple (AAPL) - Market Analysis

**Current Price:** $218.27
**Change:** -$7.14 (-3.17%)
**Status:** Down

### 📊 Analysis:
Apple is down 3.17% today — a move large enough to warrant further
investigation. After reviewing recent news, the drop appears driven
by a combination of macro pressure, geopolitical risk, and pre-event
repositioning rather than any single Apple-specific catalyst.

### 📰 Market Context:
• U.S. inflation data came in hotter than expected, reducing hopes for
  near-term rate cuts — high rates compress valuations across big tech
• Geopolitical escalation in the Middle East raised supply chain concerns,
  particularly for companies with heavy Asian manufacturing exposure
• Traders appear to be de-risking ahead of Apple's product announcement
  week — a classic "sell the news" pattern before major events
• Background concerns persist: AI feature delays vs. competitors and
  slower iPhone demand in China continue to weigh on sentiment

The price movement is largely macro and externally driven — not a signal
of a fundamental change in Apple's business.

### 💡 Key Takeaway:
Macro headwinds and geopolitical jitters drove this drop — watch how
Apple responds to its product announcements next week before drawing
any longer-term conclusions.

---
*Data as of market close. Not financial advice.*
```

---

#### 📋 Pass / Fail Checklist

**Your agent passes this scenario if:**

- ✅ Called `get_stock_price` exactly **once**
- ✅ Correctly identified the move as **greater than 2%**
- ✅ **Did call** `search_news` — this is the conditional logic working correctly
- ✅ News findings are woven into the analysis — not appended as a separate block
- ✅ News findings are **recent and relevant** to the price movement
- ✅ The analysis goes beyond company news — **macro and geopolitical factors** are considered
- ✅ The analysis explains **why** the price moved, not just **that** it moved
- ✅ Output reads as a cohesive narrative, not a data dump
- ✅ Finished in **3–4 ReAct iterations**

> ⚠️ **If your agent missed important news**, add explicit searching instructions to your prompt


---

# 🎓 Workshop Complete!

You built a **Market Intelligence Agent** — nicely done!

---

### What You Practiced

- 📈 **Real-time data fetching** with `get_stock_price`
- 🧩 **Conditional tool use** — news search only when movement > 2%
- ✏️ **Prompt engineering** — writing instructions the LLM follows precisely
- 📊 **Structured output** — formatting market analysis for readability
- 🔄 **Iterative debugging** — using test scenarios to improve your prompt
- 🛡️ **Handling edge cases** — graceful responses when things go wrong

---

### Ideas to Keep Going

- **Compare multiple stocks**: run the agent on a list of symbols and compare movements side-by-side
- **Add a chart**: use `generate_chart` to visualize price changes across a portfolio of symbols
- **Track a portfolio**: run the agent daily on a watchlist and generate a summary report
- **Set alert logic**: add a condition in the prompt to flag moves > 5% as high-priority alerts
- **Generate daily reports**: chain `get_stock_price` + `search_news` across multiple tickers automatically

**Keep building — every great agent starts with a great prompt!** 🧠
